<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/exercises/ex1/ex1_lr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML4NLP1

## Starting Point for Exercise 1, part I

This notebook is supposed to serve as a starting point and/or inspiration when starting exercise 1, part I.

One of the goals of this exercise is to get you acquainted with sklearn and related libraries like pandas and numpy. You will probably need to consult the documentation of those libraries:
- sklearn: [Documentation](https://scikit-learn.org/stable/user_guide.html)
- Pandas: [Documentation](https://pandas.pydata.org/docs/#)
- NumPy: [Documentation](https://numpy.org/doc/)
- SHAP: [Documentation](https://shap.readthedocs.io/en/latest/index.html)

## Task Description

Follow the instructions in this notebook to:

1. Explore the data and create training/test splits for your experiments

2. Build a LogisticRegression classifier and design some relevant features to apply it to your data

3. Conduct hyperparameter tuning to find the optimal hyperparameters for your model

4. Explore your model's predictions and conduct an error analysis to see where the model fails

5. Conduct an interpretability analysis, investigating the model's most important features.

6. Conduct an ablation study using a subset of languages


Throughout the notebook, there are questions that you should address in your report. These are marked with 📝❓ and are collected again in the lab report section at the end of the notebook.

☝ Note, these questions are intended to provide you with an opportunity to reflect on what it is that you are doing and the kind of challenges you might face along the way.

## Use of generative AI

AI tools may be used in this exercise to help you explain code, debug, improve documentation, and explore alternatives. Any use must be documented in the *Declaration on the use of AI* at the end of this notebook (which tools were used and for which purposes). You remain responsible for understanding, checking, and testing everything you submit, and you must be able to explain any submitted text, code, analyses, and AI-assisted contributions. See the [CL guidelines on academic integrity and AI use](https://www.cl.uzh.ch/en/Studies/Services/academic-integrity.html).

In [1]:
# Pin the library versions used in this course (the same ones as in the W03 tutorial),
# so that your notebook runs the same way for you and for your reviewers.
%pip install -q scikit-learn==1.9.1 shap==0.52.0 gdown

import sklearn, shap
print(f"scikit-learn {sklearn.__version__} | shap {shap.__version__}")

Note: you may need to restart the kernel to use updated packages.
scikit-learn 1.9.1 | shap 0.52.0


In [2]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

### Loading the datasets

In [3]:
# Download dataset
!gdown 1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs # x_train
!gdown 1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6 # x_test
!gdown 1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl # y_train
!gdown 1QaZj6bI7_78ymnN8IpSk4gVvg-C9fA6X # y_test

Downloading...
From: https://drive.google.com/uc?id=1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs
To: /Users/severinmemmishofer/PycharmProjects/ML4NLP1-2026-Tutorial-Notebooks/exercises/ex1/x_train.txt
100%|█████████████████████████████████████| 64.1M/64.1M [00:09<00:00, 6.50MB/s]
Using cookies from /Users/severinmemmishofer/.cache/gdown/cookies.txt
Downloading...
From: https://drive.google.com/uc?id=1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6
To: /Users/severinmemmishofer/PycharmProjects/ML4NLP1-2026-Tutorial-Notebooks/exercises/ex1/x_test.txt
100%|█████████████████████████████████████| 65.2M/65.2M [00:09<00:00, 6.67MB/s]
Using cookies from /Users/severinmemmishofer/.cache/gdown/cookies.txt
Downloading...
From: https://drive.google.com/uc?id=1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl
To: /Users/severinmemmishofer/PycharmProjects/ML4NLP1-2026-Tutorial-Notebooks/exercises/ex1/y_train.txt
100%|███████████████████████████████████████| 480k/480k [00:00<00:00, 7.10MB/s]
Using cookies from /Users/severinmemmishofer/.cache

In [4]:
with open('x_train.txt') as f:
    x_train = f.read().splitlines()
with open('y_train.txt') as f:
    y_train = f.read().splitlines()
with open('x_test.txt') as f:
    x_test = f.read().splitlines()
with open('y_test.txt') as f:
    y_test = f.read().splitlines()

# Reads all 4 files into python lists...

In [7]:
print(x_train[0])
print(y_train[0])

Klement Gottwaldi surnukeha palsameeriti ning paigutati mausoleumi. Surnukeha oli aga liiga hilja ja oskamatult palsameeritud ning hakkas ilmutama lagunemise tundemärke. 1962. aastal viidi ta surnukeha mausoleumist ära ja kremeeriti. Zlíni linn kandis aastatel 1949–1989 nime Gottwaldov. Ukrainas Harkivi oblastis kandis Zmiivi linn aastatel 1976–1990 nime Gotvald.
est


In [5]:
# Combine x_train and y_train into one dataframe
train_df = pd.DataFrame({'text': x_train, 'label': y_train})
# Combine x_test and y_test into one dataframe
test_df = pd.DataFrame({'text': x_test, 'label': y_test})
# Inspect the first 5 items in the train split
train_df.head()

,text,label
0,Klement Gottwaldi surnukeha palsameeriti ning ...,est
1,"Sebes, Joseph; Pereira Thomas (1961) (på eng)....",swe
2,भारतीय स्वातन्त्र्य आन्दोलन राष्ट्रीय एवम क्षे...,mai
3,"Après lo cort periòde d'establiment a Basilèa,...",oci
4,ถนนเจริญกรุง (อักษรโรมัน: Thanon Charoen Krung...,tha


In [6]:
# Get list of all labels
labels = train_df['label'].unique().tolist()
print(labels)

['est', 'swe', 'mai', 'oci', 'tha', 'orm', 'lim', 'guj', 'pnb', 'zea', 'krc', 'hat', 'pcd', 'tam', 'vie', 'pan', 'szl', 'ckb', 'fur', 'wuu', 'arz', 'ton', 'eus', 'map-bms', 'glk', 'nld', 'bod', 'jpn', 'arg', 'srd', 'ext', 'sin', 'kur', 'che', 'tuk', 'pag', 'tur', 'als', 'koi', 'lat', 'urd', 'tat', 'bxr', 'ind', 'kir', 'zh-yue', 'dan', 'por', 'fra', 'ori', 'nob', 'jbo', 'kok', 'amh', 'khm', 'hbs', 'slv', 'bos', 'tet', 'zho', 'kor', 'sah', 'rup', 'ast', 'wol', 'bul', 'gla', 'msa', 'crh', 'lug', 'sun', 'bre', 'mon', 'nep', 'ibo', 'cdo', 'asm', 'grn', 'hin', 'mar', 'lin', 'ile', 'lmo', 'mya', 'ilo', 'csb', 'tyv', 'gle', 'nan', 'jam', 'scn', 'be-tarask', 'diq', 'cor', 'fao', 'mlg', 'yid', 'sme', 'spa', 'kbd', 'udm', 'isl', 'ksh', 'san', 'aze', 'nap', 'dsb', 'pam', 'cym', 'srp', 'stq', 'tel', 'swa', 'vls', 'mzn', 'bel', 'lad', 'ina', 'ava', 'lao', 'min', 'ita', 'nds-nl', 'oss', 'kab', 'pus', 'fin', 'snd', 'kaa', 'fas', 'cbk', 'cat', 'nci', 'mhr', 'roa-tara', 'frp', 'ron', 'new', 'bar', 'ltg'


### 1.1 Exploring the training data

📝❓ Take a look at a couple of texts from different languages and answer the following questions:

1. Do you notice anything that might be challenging for the classification?
2. How is the data distributed? (i.e., how many instances per label are there in the training and test set? Is it a balanced dataset?)
3. Do you think the train/test split is appropriate (i.e., is the test data representative of the training data)? If not, please rearrange the data in a more appropriate way.


In [8]:
# TODO: Inspect the training data

# How many training examples are there pre language?
train_df['label'].value_counts()

label
est    500
swe    500
mai    500
oci    500
tha    500
      ... 
ell    500
lij    500
hau    500
mkd    500
ltz    500
Name: count, Length: 235, dtype: int64

In [9]:
# look at class distribution like in W03:

print("Class distribution:")
class_distribution = pd.DataFrame({
    'train_count': train_df['label'].value_counts(),
    'train_%': (100 * train_df['label'].value_counts(normalize=True)).round(1),
    'test_count': test_df['label'].value_counts(),
    'test_%': (100 * test_df['label'].value_counts(normalize=True)).round(1),
})
class_distribution

Class distribution:


,train_count,train_%,test_count,test_%
label,,,,
ace,500,0.4,500,0.4
afr,500,0.4,500,0.4
als,500,0.4,500,0.4
amh,500,0.4,500,0.4
ang,500,0.4,500,0.4
...,...,...,...,...
yid,500,0.4,500,0.4
yor,500,0.4,500,0.4
zea,500,0.4,500,0.4


In [10]:
class_distribution[['train_count', 'test_count']].describe()

,train_count,test_count
count,235.0,235.0
mean,500.0,500.0
std,0.0,0.0
min,500.0,500.0
25%,500.0,500.0
50%,500.0,500.0
75%,500.0,500.0
max,500.0,500.0


--> From this distribution we observe that each class/ language has EXACTLY 500 texts per language in both the training and test sets.

That means that the classes are very evenly distributed, which is a good sign; there is also no variation in number of texts per language because min = max = 500, and std. deviation is 0.0 ...

But the texts can still vary in content or length...

In [12]:
# Inspecting some of the training samples:

# have seed for the random state to have such that it is deterministic
sample = train_df.sample(5, random_state=42)

for _, row in sample.iterrows():
    print("Language:", row['label'])
    print(row['text'])
    print()

Language: ces
První písemná zmínka o obci pochází z roku 1358. Osada patřila k hradu Pořešín,pak ji Rožmberkové připojili k novohradskému panství a s ním sdílela majitele až do poloviny 19. století.

Language: ron
După ce a fost capturat pentru încercarea de asasinare a liderului Satului Noroiului, Raikage, și uciderea unor Jounini din Satul Ceții și Satul Pâmântului, Naruto Uzumaki este închis în Houzukijou, o instalație de izolare penală, de asemenea, cunoscută sub numele de Închisoarea de Sânge. Stăpânul închisorii, Mui, folosește jutsul final de închisoare pentru a fura puterea prizonierilor. În acest loc, ceva țintește viața lui Naruto Uzumaki. Lupta pentru a-și dovedi nevinovăția lui și a descoperi adevărul a început.

Language: gle
B'iad Adam Levine, Blake Shelton, Pharrell Williams agus Christina Aguilera na breithiúna don sraith seo. Bhuaigh Sawyer Fredericks an sraith agus tháinig Meghan Linsey sa dara háit.

Language: cdo
Céng-ăng-kṳ̆(晉安區) sê Hók-ciŭ-chê gì siŏh ciáh kṳ̆, iâ

In [14]:
# Print some English and German examples, since I can't read the above random languages...

examples = train_df[train_df['label'] == 'eng'].sample(3, random_state=42)

for text in examples['text']:
    print(text)
    print()

After Schmidt announced the end of his tenure as CEO on January 20, 2011, he jokingly tweeted on Twitter: "Adult-supervision no longer needed." Quartz organizational management reporter, Max Nisen, described the decade prior to Page's second appointment as Google's CEO as his "lost decade." While Page continued to exert a significant influence at Google during this time, overseeing product development and other operations, he became increasingly disconnected and less responsive over time.

Hearst's bail was revoked in May 1978 when appeals failed and the Supreme Court declined to hear her case. The prison took no special security measures for Hearst's safety until she found a dead rat on her bunk the day William and Emily Harris were arraigned for her abduction. The Harrises were convicted on a simple kidnapping charge (as opposed to the more serious kidnapping for ransom or kidnapping with bodily injury); they were released after serving a total eight years each. Although there were s

In [15]:
examples = train_df[train_df['label'] == 'deu'].sample(3, random_state=42)

for text in examples['text']:
    print(text)
    print()

Die Kirchenruine zum Heiligen Grabe, auch als Schöntalruine bekannt, ist die Ruine der 1543 bis 1545 erbauten und 1552 niedergebrannten Kirche des Beginenklosters. Es wurde vom Erzbischof Albrecht von Brandenburg um das Jahr 1500 im „Tiergarten zu Aschaffenburg“ errichtet. Zu dessen Äbtissin machte er 1540 seine Lebensgefährtin Agnes Pless. Bereits im Schmalkaldischen Krieg 1546 und wenige Jahre später im Zweiten Markgrafenkrieg 1552 wurde das Gebäude weitestgehend zerstört. Mit der Auflösung des Beginenkonvents wurde die Aschaffenburger Grabeskirche dem Verfall preisgegeben. Da das Kirchengebäude nur kurze Zeit stand, hat sich die Anwesenheit der Beginen in der Aschaffenburger Lokalgeschichte nur im Tiergarten überliefert. Die Ruine wurde später in die Parkgestaltung mit einbezogen.

Girja nahm erstmals an einer Schacholympiade 2010 mit der zweiten russischen Frauenmannschaft teil. 2014 gewann sie an der Schacholympiade der Frauen. Mit der russischen Frauenmannschaft hatte sie bereits

Some observations regarding the data:

What could be challenging is, that the text sometimes vary a lot... The german and english examples seem to be longer than the ones from the 5 random languages... But this could also be just the samples that were randomly picked...

Another factor to consider is the character set... The 'cdo' example from above, (I guess some sort of Chinese?), has different characters... Our classifier needs to be able to handle that, which could be a challenge, but maybe also work to our advantage, since such characters exclude e.g. examples with the latin

In [17]:
# Comparing Text lengths:

train_lengths = train_df['text'].str.len()
test_lengths = test_df['text'].str.len()

pd.DataFrame({
    'train': train_lengths.describe(),
    'test': test_lengths.describe()
})

,train,test
count,117500.000000,117500.000000
mean,368.954979,372.489336
std,457.201527,812.888375
min,140.000000,140.000000
25%,189.000000,189.000000
50%,271.000000,271.000000
75%,431.000000,432.000000
max,40578.000000,195402.000000


Observation: the train and test datasets seem to be very evenly matched in typical text lengths... With the exception of the max. length example in the test set, however that could be an outlier and not a general problem...

In [18]:
# Check the longest test example:

longest_idx = test_lengths.idxmax()
longest_text = test_df.loc[longest_idx, 'text']

print("Language:", test_df.loc[longest_idx, 'label'])
print("Characters:", len(longest_text))
print(longest_text[:1000])

Language: bod
Characters: 195402
འཁོར་བའི་བྱ་བཞག་སྙིང་པོ་མེད་པར་དགོངས། །སྙིང་རྗེ་ཆེན་པོས་གཞན་དོན་འབའ་ཞིག་གཉེར། །སྲིད་ཞིར་མ་ཞེན་ཐེག་ཆེན་གཞུང་བཞིན་མཛད། །མཚུངས་མེད་བླ་མའི་ཞབས་ལ་ཕྱག་འཚལ་ལོ། མགོན་པོ་ཀླུ་སྒྲུབ་ཀྱིས། མ་ཡི་ཐུག་མཐའ་རྒྱ་ཤུག་ཚིག་གུ་ཙམ། །རིལ་བུར་དྲིལ་ཀྱང་ས་ཡིས་ལོངས་མི་འགྱུར། །ཞེས་གསུངས། དེ་བཞིན་དུ་འཁོར་བ་ཐོག་མ་མེད་པ་ནས། ད་ལྟའི་བར་དུ་སྐྱེ་བ་འདི་མ་བླངས་བྱ་བ་མེད་པས་འདོད་པའི་ཕྱིར་མགོ་དང་ཡན་ལག་བཅད་པའང་གྲངས་ཀྱིས་མི་ཆོད། གྲོག་སྤུར་ལ་སོགས་སྲོག་ཆགས་ཕྲ་མོར་སྐྱེས་པའི་ཡན་ལག་རྣམས་ད་ལྟ་གཅིག་ཏུ་བསྡམ་རྒྱུ་ཡོད་ན་རི་ཡི་རྒྱལ་པོ་རི་རབ་ལས་ཀྱང་མཐོ། ཁ་ལ་ཟས་མེད་རྒྱབ་ལ་གོས་མེད། འཁྱག་ལྟོག་སྐོམ་གསུམ་ལ་སོགས་པའི་སྡུག་བསྔལ་མྱོང་ནས་ངུས་པའི་མཆི་མ་ཐམས་ཅད་མ་སྐམ་པར་བསགས་ན་མཐའི་རྒྱལ་མཚོ་ཆེན་པོ་ལས་ཀྱང་མང། དམྱལ་བའི་གནས་སུ་སྐྱེས་ཏེ་ཟངས་ཞུན་ཁོལ་མ་ཁོ་ན་རང་ཕྱོགས་ཀྱི་རྒྱ་མཚོ་ཆེན་པོ་བཞི་ལས་མང་བ་ཞིག་བཏུང་བ་ཡིན་ཀྱང། ད་དུང་འཁོར་བའི་གནས་རིས་འདི་ལ་སྐྱོ་བའི་བློ་སྐད་ཅིག་ཀྱང་མི་སྐྱེ་བར་ཞེན་ཅིང་ཆགས་པས་བཅིངས་པ་རྣམས། འཁོར་བ་མཐའ་མེད་དུ་སྔར་གྱི་སྡུག་བསྔལ་དེ་ལས་ཀྱང་མང་བར་མྱོང་དགོས་པ་ཡིན། གལ་སྲིད་བསོད་ནམས་ཆ་མཐུན་གྱི་དགེ་འབྲས་ཕྲན་བུ་རེས་ཚང

This is an outlier in length, but that is not really a reason to remove it...

But since our existing split is balanced in terms of class distribution and similar in average length, we will keep it...

### 1.2 Data preparation

Get a subset of the train/test data that includes 20 languages.
Include English (`eng`), German (`deu`), Dutch (`nld`), Danish (`dan`), Swedish (`swe`), Norwegian (either Bokmål `nob` or Nynorsk `nno`), and Japanese (`jpn`), plus 13 additional languages of your choice based on the items in the list of labels.

Note that `sklearn` classifiers accept string labels such as `'eng'` directly; the order of the classes is stored in `clf.classes_` after fitting.

In [19]:
# TODO: Create your train/test subsets of languages

# First, print labels:
print(sorted(labels))

['ace', 'afr', 'als', 'amh', 'ang', 'ara', 'arg', 'arz', 'asm', 'ast', 'ava', 'aym', 'azb', 'aze', 'bak', 'bar', 'bcl', 'be-tarask', 'bel', 'ben', 'bho', 'bjn', 'bod', 'bos', 'bpy', 'bre', 'bul', 'bxr', 'cat', 'cbk', 'cdo', 'ceb', 'ces', 'che', 'chr', 'chv', 'ckb', 'cor', 'cos', 'crh', 'csb', 'cym', 'dan', 'deu', 'diq', 'div', 'dsb', 'dty', 'egl', 'ell', 'eng', 'epo', 'est', 'eus', 'ext', 'fao', 'fas', 'fin', 'fra', 'frp', 'fry', 'fur', 'gag', 'gla', 'gle', 'glg', 'glk', 'glv', 'grn', 'guj', 'hak', 'hat', 'hau', 'hbs', 'heb', 'hif', 'hin', 'hrv', 'hsb', 'hun', 'hye', 'ibo', 'ido', 'ile', 'ilo', 'ina', 'ind', 'isl', 'ita', 'jam', 'jav', 'jbo', 'jpn', 'kaa', 'kab', 'kan', 'kat', 'kaz', 'kbd', 'khm', 'kin', 'kir', 'koi', 'kok', 'kom', 'kor', 'krc', 'ksh', 'kur', 'lad', 'lao', 'lat', 'lav', 'lez', 'lij', 'lim', 'lin', 'lit', 'lmo', 'lrc', 'ltg', 'ltz', 'lug', 'lzh', 'mai', 'mal', 'map-bms', 'mar', 'mdf', 'mhr', 'min', 'mkd', 'mlg', 'mlt', 'mon', 'mri', 'mrj', 'msa', 'mwl', 'mya', 'myv', 'm

In [20]:
selected_languages = [
    'eng', 'deu', 'nld', 'dan', 'swe', 'nob', 'jpn',
    'fra', 'ita', 'spa', 'por', 'ron',
    'fin', 'est', 'pol', 'rus', 'ukr',
    'ara', 'kor', 'zho'
]

I decided on the above selection. This gives us a nice balance of some similar languages, like roman ones (french, italian, etc...) or russian/ ukrainian, but also some choices with different alphabets such as chinese and korean.

In [21]:
# build a new subset with our selected languages:

train_subset = train_df[train_df['label'].isin(selected_languages)].copy()
test_subset = test_df[test_df['label'].isin(selected_languages)].copy()

# print shapes to check, if it worked:
print(train_subset.shape)
print(test_subset.shape)

(10000, 2)
(10000, 2)


### 2.1 Build a LogisticRegression classifier

To start with, we're going to build a very simple LogisticRegression classifier.
Use a `Pipeline` to chain together a `CountVectorizer` and a `LogisticRegression` estimator. Then perform a 5-fold cross validation and report the scores of this model as a baseline.

In [22]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# TODO: Define a very basic pipeline using a CountVectorizer and a LogisticRegression classifier

# CountVectorizer: Counting occurences of words as features.
pipe = Pipeline([
    ('vect', CountVectorizer()),
    ('clf', LogisticRegression())
])

In [23]:
# TODO: Run a cross validation to estimate the model's expected performance
from sklearn.model_selection import StratifiedKFold

# use the same function as in W03 notebook
# for 5-fold: divide train data in 5 parts; learn from 4 parts and evaluate on the fifth one
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipe,
    train_subset['text'],
    train_subset['label'],
    cv=cv,
    scoring='accuracy'
)

print("Accuracy per fold:", scores)
print("Mean accuracy:", scores.mean())

Accuracy per fold: [0.921  0.9185 0.9355 0.9275 0.9165]
Mean accuracy: 0.9238000000000002


--> Mean accuracy of roughly 92% (correctly classified texts), but the accuracies are pretty similar across the 5 folds...

(Cross validation creates separate copies of the pipeline, so the original pipe is still not fitted...)


### 2.2 Feature Engineering

So far, we've only considered the basic `CountVectorizer` at the word level to encode our input texts for our model.

Your task is to apply some text preprocessing and engineer some more informative features.

To do this, think about what other features might be relevant for determining the language of an input text.

Define a custom set of feature extractors and implement the necessary preprocessing steps to extract these features from strings.

Then initialise a processing pipeline that converts your input data into features that the model can take as input.

☝ Note, this step can be as involved as your heart desires, there is only one minimal requirement: you must use something more than the base `CountVectorizer`, and your features must be different from the document statistics shown in the W03 tutorial.


In [ ]:
# TODO: Data cleaning/Feature engineering steps

# Important: DONT use any features that we used in notebook W03
# but think/ design some other customized features!!!

In [24]:
import unicodedata

# idea for a new feature: since we saw quite different languages above, which also use different alphabets, it might be helpful as a feature to define a ratio of certain characters, such as cyrillic characters...
# although this would primarily help to distinguish russian/ ukrainian from the rest, it would still be helpful...

def cyrillic_ratio(text):
    # keep only letters and exclude punctuation, or other things...
    letters = [char for char in text if char.isalpha()]

    if len(letters) == 0:
        return 0.0

    n_cyrillic = sum(
        # gets the name of a character
        'CYRILLIC' in unicodedata.name(char, '')
        for char in letters
    )

    return n_cyrillic / len(letters)

In [25]:
# test the function:
print(cyrillic_ratio("Hello!"))
print(cyrillic_ratio("Привет!"))
print(cyrillic_ratio("Hello Привет!"))
print(cyrillic_ratio("123!"))

0.0
1.0
0.5454545454545454
0.0


In [26]:
from sklearn.base import BaseEstimator, TransformerMixin

# make a new class, similar to "TextStats" in W03
class ScriptFeatures(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        return self # again nothing to learn

    def transform(self, X):
        return np.asarray([
            [cyrillic_ratio(text)] for text in X # calculates our ratio per text
        ], dtype=float)

    def get_feature_names_out(self, input_features=None):
        return np.array(['cyrillic_ratio'])

In [27]:
# test it:
script_features = ScriptFeatures()
print(script_features.fit_transform(["Hello!", "Привет!"]))

[[0.]
 [1.]]


In [28]:
from sklearn.pipeline import FeatureUnion

# combine the features with CountVectorizer using FeatureUnion, similar to the W03 notebook...
features = FeatureUnion([
    ('vect', CountVectorizer()),
    ('script', ScriptFeatures())
])

pipe_features = Pipeline([
    ('features', features),
    ('clf', LogisticRegression())
])

In [29]:
# test cross-validation again, similar to the baseline:

scores_features = cross_val_score(
    pipe_features,
    train_subset['text'],
    train_subset['label'],
    cv=cv,
    scoring='accuracy'
)

print("Accuracy per fold:", scores_features)
print("Baseline mean accuracy:", scores.mean())
print("With Cyrillic ratio:", scores_features.mean())
print(
    "Difference in percentage points:",
    100 * (scores_features.mean() - scores.mean())
)

Accuracy per fold: [0.9285 0.922  0.943  0.933  0.9205]
Baseline mean accuracy: 0.9238000000000002
With Cyrillic ratio: 0.9293999999999999
Difference in percentage points: 0.5599999999999716


So, adding the proportion of Cyrillic letters improved the accuracy in the same 5 folds as used in the baseline.

The improvement was not very large, but it was seen across all of the 5 folds at least...

--> From 92.38 to 92.94% or an increase of 0.56 % points...

In [30]:
# generalise the function to other alphabets rather than just cyrillic

def script_ratio(text, name_part):
    letters = [char for char in text if char.isalpha()]

    if len(letters) == 0:
        return 0.0

    n_matching = sum(
        name_part in unicodedata.name(char, '')
        for char in letters
    )

    return n_matching / len(letters)

In [31]:
print(script_ratio("Привет!", "CYRILLIC"))
print(script_ratio("مرحبا", "ARABIC"))
print(script_ratio("안녕하세요", "HANGUL"))

1.0
1.0
1.0


In [34]:
# similar to the Class above, but with other script ratios...

class MultiScriptFeatures(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray([
            [
                script_ratio(text, 'CYRILLIC'),
                script_ratio(text, 'ARABIC'),
                script_ratio(text, 'HANGUL'),
                script_ratio(text, 'CJK UNIFIED IDEOGRAPH'),
                script_ratio(text, 'HIRAGANA'),
                script_ratio(text, 'KATAKANA'),
            ]
            for text in X
        ], dtype=float)

    def get_feature_names_out(self, input_features=None):
        return np.array([
            'cyrillic_ratio',
            'arabic_ratio',
            'hangul_ratio',
            'han_ratio',
            'hiragana_ratio',
            'katakana_ratio',
        ])

In [35]:
script_features = MultiScriptFeatures()

print(script_features.get_feature_names_out())
print(script_features.fit_transform([
    "Привет", "مرحبا", "안녕", "中文", "こんにちは", "カタカナ"
]))

['cyrillic_ratio' 'arabic_ratio' 'hangul_ratio' 'han_ratio'
 'hiragana_ratio' 'katakana_ratio']
[[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]]


In [36]:
#build new pipeline

features_multi = FeatureUnion([
    ('vect', CountVectorizer()),
    ('script', MultiScriptFeatures())
])

pipe_multi = Pipeline([
    ('features', features_multi),
    ('clf', LogisticRegression())
])

In [37]:
# evaluate it again...

scores_multi = cross_val_score(
    pipe_multi,
    train_subset['text'],
    train_subset['label'],
    cv=cv,
    scoring='accuracy'
)

print("Accuracy per fold:", scores_multi)
print("Baseline:", scores.mean())
print("Cyrillic only:", scores_features.mean())
print("Six script features:", scores_multi.mean())

Accuracy per fold: [0.974  0.9715 0.9825 0.9765 0.9715]
Baseline: 0.9238000000000002
Cyrillic only: 0.9293999999999999
Six script features: 0.9752000000000001


So, it seems that adding the function also for other scripts increased the Accuracy to a total of 97.5%. It might be that the alphabets produce useful information, even if some words were not seen during training (for jus count-vectorizer).

---

### 3.1 Grid Search

Use sklearn's GridSearchCV and experiment with the following hyperparameters:
1. Regularization: its strength `C` and its type via `l1_ratio` (`l1_ratio=0` is L2, `l1_ratio=1` is L1)
2. Solver (note that not every solver supports every regularization type)
3. Experiment with parameters of the Vectorizer (optional, but highly advised)

☝ Note, don't overdo it at the beginning, since runtime might go up fast!

Make sure you read through the [docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) to get an understanding of what these parameters do.


In [38]:
# Find default parameters of our Logistic Regression Classifier:
params = pipe_multi.get_params()

print("C:", params['clf__C'])
print("L1 ratio:", params['clf__l1_ratio'])
print("Solver:", params['clf__solver'])

C: 1.0
L1 ratio: 0.0
Solver: lbfgs


In [39]:
# TODO: GridSearchCV

param_grid = [
    {
        'clf__solver': ['lbfgs'], # "default solver"
        'clf__C': [0.1, 1.0], # Stronger or weaker regularization parameter C
        'clf__l1_ratio': [0.0], # L2 regularization (discourages large weights)
    },
    {
        'clf__solver': ['saga'], # 2nd option of a solver...
        'clf__C': [0.1, 1.0], # Stronger or weaker regularization parameter C
        'clf__l1_ratio': [0.0, 1.0], # Try L1 AND L2 Regularization separately (L1 prefers weights exactly 0)
    },
]

# could also try our own solvers, but it's up to us...
# DONT overdo the number of params for Grid Search such that it does not take too long...!

# in this combination: 2 combinations for lbfgs solver (depending on C)
# + 4 combinations of saga solver --> 6 in total
# * 5 folds --> 6*5 = 30 runs!

In [40]:
from sklearn.model_selection import GridSearchCV

# execute the Grid Search
pipe_multi.set_params(
    clf__max_iter=1000,
    clf__random_state=42
)

grid = GridSearchCV(
    pipe_multi,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=2,
    verbose=2
)

In [41]:
import time

t0 = time.time()

grid.fit(
    train_subset['text'],
    train_subset['label']
)

print(f"Grid search took {time.time() - t0:.0f} seconds")
print("Best parameters:", grid.best_params_)
print("Best mean CV accuracy:", grid.best_score_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV] END ...clf__C=0.1, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  10.9s
[CV] END ...clf__C=0.1, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  11.4s
[CV] END ...clf__C=0.1, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  11.6s
[CV] END ...clf__C=0.1, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  11.8s
[CV] END ...clf__C=0.1, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  11.3s
[CV] END ...clf__C=1.0, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  11.6s
[CV] END ...clf__C=1.0, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  12.1s
[CV] END ...clf__C=1.0, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  11.8s
[CV] END ...clf__C=1.0, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  11.9s
[CV] END ...clf__C=1.0, clf__l1_ratio=0.0, clf__solver=lbfgs; total time=  11.6s
[CV] END ....clf__C=0.1, clf__l1_ratio=0.0, clf__solver=saga; total time=  29.7s
[CV] END ....clf__C=0.1, clf__l1_ratio=0.0, clf__

/Users/severinmemmishofer/PycharmProjects/ML4NLP1-2026-Tutorial-Notebooks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ....clf__C=0.1, clf__l1_ratio=1.0, clf__solver=saga; total time= 1.2min


/Users/severinmemmishofer/PycharmProjects/ML4NLP1-2026-Tutorial-Notebooks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ....clf__C=0.1, clf__l1_ratio=1.0, clf__solver=saga; total time= 1.2min


...

/Users/severinmemmishofer/PycharmProjects/ML4NLP1-2026-Tutorial-Notebooks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ....clf__C=1.0, clf__l1_ratio=1.0, clf__solver=saga; total time= 5.2min


/Users/severinmemmishofer/PycharmProjects/ML4NLP1-2026-Tutorial-Notebooks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ....clf__C=1.0, clf__l1_ratio=1.0, clf__solver=saga; total time= 5.1min
Grid search took 1250 seconds
Best parameters: {'clf__C': 1.0, 'clf__l1_ratio': 0.0, 'clf__solver': 'saga'}
Best mean CV accuracy: 0.9764999999999999


/Users/severinmemmishofer/PycharmProjects/ML4NLP1-2026-Tutorial-Notebooks/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


The saga solver with the L1 Regularization took quite a lot longer, sometimes 5 minutes per fold fit, especially with a weaker Regularization of C=1.0... (C is the inverse of regularization strength...)

The convergence warnings just mean that the max. number of iterations was reached, but it might still produce a meaningful result; it just hasn't fully converged in that amount of iterations...

### 3.2 Best Model Selection

After conducting our Grid Search, we should be able to identify our best model by inspecting the Grid Search result attribute `cv_results_`. (Hint: `cv_results_` is a dictionary, so convert it to a Pandas Dataframe for easy inspection.)

📝❓ What were the hyperparameters of your best-performing model according to cross-validation?

📝❓ What is the advantage of grid search cross-validation?


In [43]:
# TODO: Select the best model based on the GridSearch results

results = pd.DataFrame(grid.cv_results_)

columns = [
    'param_clf__solver',
    'param_clf__C',
    'param_clf__l1_ratio',
    'mean_test_score',
    'std_test_score',
    'mean_fit_time',
    'rank_test_score'
]

results[columns].sort_values('rank_test_score')

,param_clf__solver,param_clf__C,param_clf__l1_ratio,mean_test_score,std_test_score,mean_fit_time,rank_test_score
4,saga,1.0,0.0,0.9765,0.003362,37.612146,1
2,saga,0.1,0.0,0.9755,0.004147,26.218880,2
1,lbfgs,1.0,0.0,0.9752,0.004094,10.597608,3
5,saga,1.0,1.0,0.9692,0.004718,281.565138,4
0,lbfgs,0.1,0.0,0.9686,0.004510,10.205224,5
3,saga,0.1,1.0,0.9425,0.007733,69.895182,6


- As we can see in the table above, the parameters that worked best in our case were the saga solver with C=1.0 and L2 regularization, followed closely by the same setup with C=0.1, so stronger regularization. The best score was 97.65% accuracy.
- After that, the lbfgs solver with the same parameter settings as our number 1 configuration followed, which was also faster
- Also: some SAGA (Stochastic Average Gradient Accelerated) fits reached the iteration limit
- The advantage of using grid-search cross-validation is that it systematically compares parameter settings on the same sets of training and validation data for a better comparison amongst each other. Each config. uses the same 5 folds and the validation scores get averaged.

### 3.3 Model Evaluation

Once you have identified your best model, use it to predict the languages of texts in the test split.

📝❓ According to standard metrics (e.g. Accuracy, Precision, Recall and F1), how well does your model perform on the held-out test set?


In [ ]:
# TODO: Evaluate the model by inspecting the predictions on the held-out test set

---

### 4.1 Error Analysis

Inspect your model's predictions using a confusion matrix and provide a summary of what you find in your report.

📝❓ Where does your model do well and where does it fail?

📝❓ What are some possible reasons for why it fails in these cases?

In [ ]:
# TODO: Inspect the model's predictions on the different classes

---

### 5.1 Interpretability Analysis

Now that you have your best model, it's time to dive deep into understanding how the model makes predictions.

It is important that we can explain and visualise our models to improve task performance. Explainable models help characterise model fairness, transparency, and outcomes.

Let's try to understand what our best-performing logistic regression classification model has learned.

Inspect the 20 most important features for the languages English, Swedish, Norwegian (whichever of `nob`/`nno` you included), and Japanese. Please make sure that the features are named and human-interpretable, not things like "Feat_1". (Hint: if you have used custom feature extractors in your pipeline, you may need to adapt these to make sure that the feature names are maintained.)

📝❓ What is more important, extra features or the outputs of the vectorizer? Please discuss.

We recommend using the [SHAP library](https://shap.readthedocs.io/en/latest/example_notebooks/tabular_examples/linear_models/Sentiment%20Analysis%20with%20Logistic%20Regression.html) as discussed in the [W03 tutorial](https://github.com/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/tutorials_notebooks_in_class_2026/W03_pipeline_gridsearch_shap.ipynb), which shows SHAP for a multi-class linear model.

☝ Note, if you prefer to use another interpretability tool, we will accept answers from any explanation library/method as long as the explanations for the model weights are provided in a structured/clear way.



In [ ]:
# shap was installed and imported in the setup cell at the top of the notebook
import shap

In [ ]:
# TODO: Inspect most important features according to the model using SHAP

---

### 6.1 Ablation Study

Lastly, we want to conduct a small ablation study to investigate how well our model performs under different conditions.

As a first step, choose the two languages for which the classifier worked best.

Next, re-fit the best model six times, each time reducing the **length** of the texts. To do this, create a custom `TextReducer` class that truncates every text to at most `max_len` characters and include it as a preprocessing step in your pipeline, so that the truncation is applied both when fitting and when predicting. The class should take `max_len` as a hyperparameter that can be set to train the following models:

- Model 1: `max_len = None` (i.e. no truncation!)
- Model 2: `max_len = 500`
- Model 3: `max_len = 250`
- Model 4: `max_len = 150`
- Model 5: `max_len = 100`
- Model 6: `max_len = 50`

Use average accuracy over the cross validation scores for each model to measure performance for each ablation setting.

📝❓ How does reducing the length of the training texts affect the performance of the classifier? And what could be some possible reasons for this?

In [ ]:
# TODO: Ablation study

# better to answer any questions and put them in the Lab Report at the end!!

# + AI declaration at the end!!

# also declare, if we use AI also to write code, and such things...
# even if AI writes code for us; also try to understand what it does and try to understand the overall picture!!

# + submit 1 document per group!! And what about report..?
# create 1 IDENTICAL document for all team-members!

# For this exercise: 2 parts/ 2 notebooks!! Have to create an Archive to upload!!
# 2nd part of the exercise will be updated in the REPO this week...!

---

## Lab report

📝❓ Write your lab report here, addressing all questions in the notebook. For convenience, the questions are collected below; please keep your answers clearly marked.

### Questions

**1.1 Exploring the training data**

- 📝❓ Do you notice anything that might be challenging for the classification?
- 📝❓ How is the data distributed? (i.e., how many instances per label are there in the training and test set? Is it a balanced dataset?)
- 📝❓ Do you think the train/test split is appropriate (i.e., is the test data representative of the training data)? If not, please rearrange the data in a more appropriate way.

**3.2 Best Model Selection**

- 📝❓ What were the hyperparameters of your best-performing model according to cross-validation?
- 📝❓ What is the advantage of grid search cross-validation?

**3.3 Model Evaluation**

- 📝❓ According to standard metrics (e.g. Accuracy, Precision, Recall and F1), how well does your model perform on the held-out test set?

**4.1 Error Analysis**

- 📝❓ Where does your model do well and where does it fail?
- 📝❓ What are some possible reasons for why it fails in these cases?

**5.1 Interpretability Analysis**

- 📝❓ What is more important, extra features or the outputs of the vectorizer? Please discuss.

**6.1 Ablation Study**

- 📝❓ How does reducing the length of the training texts affect the performance of the classifier? And what could be some possible reasons for this?

## Declarations

Please fill in both declarations below by editing this cell (double-click it). Mark the applicable option by replacing ☐ with ☒ (copy the symbol from here: ☒) and complete the text field where required. Your reviewers will check that both declarations are present and completed.

### Contribution declaration

By submitting this work, the group declares that the contributions of the group members were:

☐ approximately equal

☐ not approximately equal

If the contributions were not approximately equal, no individual group member needs to be identified.

### Declaration on the use of AI

The group declares that any use of AI-based tools in preparing this work complied with the rules specified for this exercise.

☐ No AI-based tools were used.

☐ AI-based tools were used in accordance with the rules of the exercise.

If AI-based tools were used, briefly state the tool(s) used and the purpose(s) for which they were used:

